# 02 — Fine-Tune RF-DETR (Phase 3, 4–8 credits)

Teaches the detector one new class (`forklift`) and adapts it to your cameras.
Starting from COCO weights is why ~500 images suffice instead of ~100,000: the
pretrained weights already encode generic visual features, so you are not
learning vision from scratch.

**Before running:** confirm the train/valid split was made **by source video**,
not randomly by frame. Frames a second apart are near-duplicates; a random split
leaks them across both sides and produces beautiful, fictional metrics that
collapse on new footage (§4.2).

In [ ]:
# Colab setup. T4 GPU ONLY — never A100 (2 credits/hr vs 12).
!nvidia-smi --query-gpu=name,memory.total --format=csv

# Pins match requirements.txt; see src/detector.py for the API drift each guards.
!pip install -q "rfdetr>=1.9.0,<2.0.0" "supervision>=0.29,<0.30" trackers rtmlib onnxruntime-gpu

from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/warehouse-safety'

import sys
sys.path.insert(0, PROJECT)          # so `from src...` resolves to the mirrored repo
print('project:', PROJECT)

In [ ]:
# Copy the dataset to LOCAL Colab disk. Reading thousands of images through the
# Drive mount is 10-50x slower and will dominate training time (§1.4).
!cp -r "{PROJECT}/data/dataset" /content/dataset
!ls /content/dataset

In [ ]:
# Read class IDs from the dataset — never hardcode them. Roboflow sometimes
# inserts a dummy category 0, shifting everything (§4.4).
!cd {PROJECT} && python -m scripts.discover_class_ids /content/dataset

In [ ]:
import torch
from rfdetr import RFDETRBase

model = RFDETRBase()          # COCO weights = transfer learning starting point

model.train(
    dataset_dir='/content/dataset',
    epochs=25,
    batch_size=4,             # T4-safe
    grad_accum_steps=4,       # effective batch 16
    lr=1e-4,
    output_dir=f'{PROJECT}/models/rfdetr_v1',   # checkpoints straight to Drive
)

### If it fails or underperforms

| Symptom | Fix |
|---|---|
| CUDA out of memory | `batch_size=2, grad_accum_steps=8` (same effective batch) |
| Small/distant people missed | `RFDETRBase(resolution=728)` — must be divisible by 56; more compute |
| Person mAP dropped vs notebook 01 | Catastrophic forgetting → retrain with `lr=5e-5` |
| Forklift mAP low | Almost always a **data** problem. Inspect annotated val images and fix labels/counts before touching any hyperparameter |

Expect **1.5–3 h on a T4** for ~500 images. Validation mAP prints each epoch.
Best weights are `checkpoint_best_ema.pth` (EMA = smoothed weights — use this one).

In [ ]:
# Verify the fine-tuned model visually.
import glob, os, cv2, json, supervision as sv
from src.detector import RFDetrDetector

WEIGHTS = f'{PROJECT}/models/rfdetr_v1/checkpoint_best_ema.pth'
detector = RFDetrDetector(weights=WEIGHTS, threshold=0.5)

# After fine-tuning, class_id indexes YOUR dataset's categories, not COCO's 80.
cats = json.load(open('/content/dataset/valid/_annotations.coco.json'))['categories']
CLASS_NAMES = {c['id']: c['name'] for c in cats}
print('classes:', CLASS_NAMES)

OUT = f'{PROJECT}/outputs/val'
os.makedirs(OUT, exist_ok=True)
box_ann, lab_ann = sv.BoxAnnotator(), sv.LabelAnnotator()

for path in sorted(glob.glob('/content/dataset/valid/*.jpg'))[:20]:
    bgr = cv2.imread(path)
    dets = detector(bgr)
    labels = [f"{CLASS_NAMES.get(int(c), '?')} {conf:.2f}"
              for c, conf in zip(dets.class_id, dets.confidence)]
    cv2.imwrite(f'{OUT}/val_{os.path.basename(path)}',
                lab_ann.annotate(box_ann.annotate(bgr.copy(), dets), dets, labels))
print(f'annotated validation images -> {OUT}')

## Acceptance check

Final validation **mAP50 ≥ 0.80 person, ≥ 0.60 forklift** (PoC bar; 0.8+ forklift
is normal with ~500 good images).

Record the measured numbers — they go straight into the final write-up, and
"measured mAP" is one of the six items in the definition of done.

Next: Phase 5 (tracking) and Phase 6 (calibration), then run the pipeline:

```bash
python -m src.run_pipeline \
  --video data/raw_videos/test_clip.mp4 \
  --calib data/calibration/cam1.json \
  --weights models/rfdetr_v1/checkpoint_best_ema.pth \
  --person-id 2 --forklift-id 1
```

Then **Runtime → Disconnect**.